# Harris County, TX — 3DEP DEM elevation sampling (Phase 1, step 3)

Third link. Download the four USGS 3DEP 1/3 arc-second (~10 m) tiles that
cover the Harris County bbox, mosaic them, and sample ground elevation at
every building. Output is a tiny parquet keyed by `id` so the depth notebook
can join it back without re-reading the building geometries.

**Sampling rule.** For small footprints (≤ 500 m²) we take the centroid;
for larger ones we sample every vertex and keep the **minimum** — the most
conservative ground level under the building, which is what HAZUS depth
curves implicitly assume. Footprint area is computed in UTM zone 15N
(EPSG:32615), the appropriate projection for east Texas.

**Datums.** 3DEP 1/3” is NAVD88 vertical, meters. NFHL BFE is NAVD88, feet.
We keep elevation in meters here; the depth notebook does the foot→meter
conversion on the BFE side.

**One-time install** (rasterio is not in the base venv):
```
uv pip install rasterio tqdm
```

**Inputs:** `data/raw/harris_buildings.parquet` from notebook 01; 3DEP tiles
from the public USGS S3 bucket (anonymous HTTPS).

**Outputs:**
- `data/raw/dem_tiles/USGS_13_n{NN}w{WWW}.tif` — cached tiles (~50 MB each)
- `data/raw/harris_building_elev.parquet` — `id`, `elev_m`


In [ ]:
from pathlib import Path

import httpx
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.merge import merge as rio_merge
from shapely.geometry import Point
from tqdm import tqdm


In [ ]:
MIN_LON, MAX_LON = -95.91, -94.90
MIN_LAT, MAX_LAT = 29.49, 30.18

# Footprints ≥ this area (m² in UTM 15N) get vertex sampling instead of
# centroid only. ~500 m² ≈ a large single-family home; anything bigger is
# likely to span enough terrain that the centroid lies on a different
# elevation than the lowest opening.
VERTEX_SAMPLE_THRESHOLD_M2 = 500.0
UTM_CRS = "EPSG:32615"  # UTM zone 15N covers Harris County

# 3DEP 1/3" tiles are 1°×1°, named by NW corner integer (n{lat+1}w{|lon|+1}
# is wrong — the NW corner of n30w096 is at lat 30, lon -96, covering
# lat 29–30 / lon -96–-95).
TILE_BUCKET = "https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/current"

REPO_ROOT = Path.cwd().resolve().parents[1]
RAW_DIR = REPO_ROOT / "data" / "raw"
DEM_DIR = RAW_DIR / "dem_tiles"
BUILDINGS_PATH = RAW_DIR / "harris_buildings.parquet"
ELEV_PATH = RAW_DIR / "harris_building_elev.parquet"
DEM_DIR.mkdir(parents=True, exist_ok=True)
ELEV_PATH


## Enumerate and download 3DEP tiles

Walk integer lat/lon corners that cover the bbox; for each tile, name it the
USGS way (`n{NW_lat}w{|NW_lon|}`, both zero-padded) and download once if not
already cached. Tiles are ~50 MB each as Cloud Optimized GeoTIFFs.


In [ ]:
def tiles_for_bbox(min_lat: float, max_lat: float, min_lon: float, max_lon: float) -> list[tuple[int, int]]:
    """Return list of (nw_lat, nw_lon) integer corners for 1° tiles overlapping the bbox."""
    nw_lats = range(int(np.floor(min_lat)) + 1, int(np.ceil(max_lat)) + 1)
    nw_lons = range(int(np.floor(min_lon)), int(np.ceil(max_lon)))
    return [(lat, lon) for lat in nw_lats for lon in nw_lons]


def tile_name(nw_lat: int, nw_lon: int) -> str:
    return f"n{nw_lat:02d}w{abs(nw_lon):03d}"


tile_corners = tiles_for_bbox(MIN_LAT, MAX_LAT, MIN_LON, MAX_LON)
print(f"need {len(tile_corners)} tiles: {[tile_name(*c) for c in tile_corners]}")


In [ ]:
def download_tile(nw_lat: int, nw_lon: int) -> Path:
    name = tile_name(nw_lat, nw_lon)
    local = DEM_DIR / f"USGS_13_{name}.tif"
    if local.exists() and local.stat().st_size > 0:
        return local
    url = f"{TILE_BUCKET}/{name}/USGS_13_{name}.tif"
    print(f"  downloading {url}")
    with httpx.stream("GET", url, timeout=180.0, follow_redirects=True) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with local.open("wb") as fh, tqdm(total=total, unit="B", unit_scale=True, leave=False) as bar:
            for chunk in resp.iter_bytes(chunk_size=1 << 20):
                fh.write(chunk)
                bar.update(len(chunk))
    return local


tile_paths = [download_tile(*c) for c in tile_corners]
tile_paths


## Mosaic and clip to bbox

Merge the four tiles into a single in-memory raster, then clip to the
Harris bbox (with a small padding) so the sampling step doesn't touch any
cells we don't need. Stays well under a GB so this is fine in memory.


In [ ]:
srcs = [rasterio.open(p) for p in tile_paths]
mosaic, mosaic_transform = rio_merge(srcs, bounds=(MIN_LON - 0.01, MIN_LAT - 0.01, MAX_LON + 0.01, MAX_LAT + 0.01))
mosaic_crs = srcs[0].crs
mosaic_nodata = srcs[0].nodata
for s in srcs:
    s.close()
print(f"mosaic shape: {mosaic.shape}, crs: {mosaic_crs}, nodata: {mosaic_nodata}")
print(f"min/max elev (m): {np.nanmin(mosaic[mosaic != mosaic_nodata]):.2f} / {np.nanmax(mosaic[mosaic != mosaic_nodata]):.2f}")


## Sample elevation at every building

Two passes:

1. **Small** (`area_m2 ≤ 500`): a single sample at the centroid.
2. **Large** (`> 500`): sample at every exterior vertex, take the minimum.

Both pass coordinates straight to `rasterio.sample` in mosaic CRS
(NAD83, EPSG:4269 — functionally identical to WGS84 at meter precision so
we feed lon/lat directly).


In [ ]:
buildings = gpd.read_parquet(BUILDINGS_PATH)
buildings = buildings.set_crs("EPSG:4326", allow_override=True)
buildings["area_m2"] = buildings.to_crs(UTM_CRS).geometry.area
buildings["centroid"] = buildings.geometry.centroid
len(buildings), buildings["area_m2"].describe()


In [ ]:
def sample_at(coords: list[tuple[float, float]]) -> np.ndarray:
    """Sample the in-memory mosaic at a list of (lon, lat) tuples; nodata → NaN."""
    rows, cols = rasterio.transform.rowcol(mosaic_transform, [c[0] for c in coords], [c[1] for c in coords])
    rows = np.clip(np.asarray(rows), 0, mosaic.shape[1] - 1)
    cols = np.clip(np.asarray(cols), 0, mosaic.shape[2] - 1)
    vals = mosaic[0, rows, cols].astype("float64")
    if mosaic_nodata is not None:
        vals = np.where(vals == mosaic_nodata, np.nan, vals)
    return vals


small_mask = buildings["area_m2"].fillna(0) <= VERTEX_SAMPLE_THRESHOLD_M2
elev = np.full(len(buildings), np.nan, dtype="float64")

small = buildings.loc[small_mask, "centroid"]
elev[small_mask.values] = sample_at(list(zip(small.x.values, small.y.values)))
print(f"sampled {small_mask.sum():,} small footprints at centroids")

large_idx = np.flatnonzero(~small_mask.values)
for i in tqdm(large_idx, desc="vertex sample"):
    geom = buildings.geometry.iloc[i]
    if geom.geom_type == "Polygon":
        coords = list(geom.exterior.coords)
    elif geom.geom_type == "MultiPolygon":
        coords = [c for p in geom.geoms for c in p.exterior.coords]
    else:
        coords = [(geom.centroid.x, geom.centroid.y)]
    vals = sample_at(coords)
    if np.all(np.isnan(vals)):
        continue
    elev[i] = float(np.nanmin(vals))
print(f"sampled {len(large_idx):,} large footprints at vertices (min)")


In [ ]:
out = buildings[["id"]].copy()
out["elev_m"] = elev
out.to_parquet(ELEV_PATH, compression="zstd")
n_missing = int(out["elev_m"].isna().sum())
print(f"wrote {ELEV_PATH} — {len(out):,} rows, {n_missing:,} with no elevation")
out["elev_m"].describe()


## Next

Move to `04_harris_county_depths.ipynb`: spatial-join buildings to NFHL
zones, fill missing static BFE from the nearest `S_BFE` line, and compute
100-yr / 500-yr inundation depths per building.
